# 07 -- CNN SFT (Step 14b) and CNN RL from CNN SFT (Step 18), one chain

**Step 14b:** AlphaZero's network (6 x 64) trained on the 500k Stockfish labels, batch 256, 2 passes, a checkpoint
at the end of each pass. **Step 18:** AlphaZero's self-play training, settings unchanged, started from CNN SFT pass 1
and run until total training compute reaches 19.3 GPU-h (as the LLM's final models).

Order (plan v5, Derek Nov 26): train CNN SFT -> **auto-check** (validation move loss fell, and pass 1 without search
beats `random` on the panel; otherwise the notebook stops, so RL is never started on a broken model) -> rate CNN SFT
pass 1 and pass 2, each with and without search (50 sims), on the full panel -> RL -> rate its final checkpoint both
ways. Every stage resumes where it stopped.

`SMOKE=1` (from the shell): a tiny SFT (4k positions) and 2 tiny RL rounds in separate `_smoke` folders, no evals --
checks the code runs end to end on the pod. Needs a GPU pod.

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import json
import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
print("GPU:", GPU)

## Settings
CNN SFT from `configs/cnn_sft.yaml`; the network and the RL / search settings from `configs/alphazero.yaml` (unchanged);
the panel from `configs/eval.yaml`.

In [ ]:
import dataclasses
from eval.anchors import ResultsTable
from eval.panel import load_panel
from train.az_train import load_configs, run_name as az_run_name
from train.cnn_sft import CNNSFTConfig, run_name

SMOKE = os.environ.get("SMOKE", "0") == "1"
RL_TOTAL_GPU_H = 19.3          # CNN SFT pass 1 + RL (as the LLM's final models: 19.27 and 19.58)
SEARCH_SIMS = 50               # AlphaZero's
eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
data_cfg = yaml.safe_load(open("configs/data.yaml"))
GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]
PANEL = load_panel()
AZ_CFG, NET, MCTS, _ = load_configs("configs/alphazero.yaml")
CFG = CNNSFTConfig.from_dict(yaml.safe_load(open("configs/cnn_sft.yaml")))
if SMOKE:
    CFG = dataclasses.replace(CFG, train_positions=4096, val_positions=512, tag="smoke")
RUN = run_name(CFG, NET)
RUN_DIR = os.path.join(ROOT, "checkpoints", "cnn_sft", RUN)
DATA_DIR = os.path.join(ROOT, "data", "processed", CFG.dataset)
LABEL_CPU = json.load(open(os.path.join(DATA_DIR, "manifest.json")))["stockfish_cpu_seconds"] / 3600
results_path = os.path.join(ROOT, eval_cfg["results_file"])
print("smoke:", SMOKE, "| CNN SFT run:", RUN, "| net:", NET, "| search sims:", SEARCH_SIMS)

## 1. Train CNN SFT (Step 14b)
Positions are encoded once into AlphaZero's buffer format (a few minutes for 500k), then 2 passes at batch 256.
Skipped if both passes are already saved.

In [ ]:
import time
from data.dataset import load_split
from train.cnn_sft import encode_labels, list_checkpoints as sft_checkpoints, train as train_sft, val_rows

if len(sft_checkpoints(RUN_DIR)) >= CFG.epochs:
    print("CNN SFT already trained:", [p for p, _ in sft_checkpoints(RUN_DIR)])
else:
    train_df, val_df = load_split(os.path.join(ROOT, "data", "processed"), CFG.dataset, data_cfg["val_fraction"],
                                  data_cfg["seed"], val_from=CFG.val_from)
    if CFG.train_positions:
        train_df = train_df.sample(n=min(CFG.train_positions, len(train_df)), random_state=CFG.seed)
    if CFG.val_positions:
        val_df = val_df.sample(n=min(CFG.val_positions, len(val_df)), random_state=CFG.seed)
    t0 = time.perf_counter()
    train_data, val_data = encode_labels(train_df), encode_labels(val_df)
    print(f"encoded {len(train_data[3]):,} train / {len(val_data[3]):,} validation positions "
          f"in {time.perf_counter() - t0:.0f}s")
    summary = train_sft(CFG, NET, train_data, val_data, RUN_DIR, extra_info={"gpu": GPU})
    print(json.dumps(summary, indent=2))
    del train_data, val_data
for r in val_rows(RUN_DIR):
    print(r)

## 2. Auto-check, then rate CNN SFT on the panel
The check: validation move loss after pass 1 is below its value before training, **and** pass 1 without search rates
above `random`. Either failure raises an error, which stops the notebook before RL. Then the other three CNN SFT rows.

In [ ]:
from eval.agents import agent_name
from eval.anchors import build_reference_agents, close_agents
from eval.panel import extend, fill_rung_wdl
from train.cnn_sft import gpu_hours_at as sft_gpu_hours_at

def sft_spec(p, sims):
    name = f"{RUN}@p{p}" if sims else f"{RUN}_nosearch@p{p}"
    run = f"cnn_sft/{RUN}" if sims else f"cnn_sft/{RUN}_nosearch"
    fields = {"run": run, "step": p, "seed": 0, "train_gpu_hours": round(sft_gpu_hours_at(RUN_DIR, p), 5),
              "label_cpu_hours": LABEL_CPU, "dataset_size": json.load(open(os.path.join(RUN_DIR, "run_config.json")))["train_rows"],
              "note": f"CNN SFT (Step 14b), pass {p}, " + (f"{sims}-sim search at play" if sims else "no search")}
    return {"name": name, "kind": "cnn_sft", "run": RUN, "step": p, "simulations": sims}, fields

v = {r["pass"]: r for r in val_rows(RUN_DIR)}
learned = v[1]["val_move_loss"] < v[0]["val_move_loss"]
print(f"validation move loss: {v[0]['val_move_loss']:.3f} before -> {v[1]['val_move_loss']:.3f} after pass 1 "
      f"(top-1 {v[0]['val_top1']:.3f} -> {v[1]['val_top1']:.3f})")
if not learned:
    raise RuntimeError("auto-check failed: CNN SFT's validation move loss did not fall -- RL not started")

if SMOKE:
    print("smoke: evals skipped")
else:
    results = ResultsTable(results_path)
    fill_rung_wdl(results, PANEL[4:])
    opponents = build_reference_agents(results, PANEL, ROOT)
    try:
        spec, fields = sft_spec(1, 0)
        print(f"\n== {spec['name']}")
        extend(spec, results, ROOT, opponents, games_per_opponent=GAMES, batch_games=BATCH, fields=fields, gpu=GPU)
        rating = float(ResultsTable(results_path).rows[spec["name"]]["rating"])
        floor = float(ResultsTable(results_path).rows["random"]["rating"])
        print(f"auto-check: pass 1 without search {rating:.0f} vs random {floor:.0f}")
        if rating <= floor:
            raise RuntimeError("auto-check failed: CNN SFT pass 1 doesn't beat random -- RL not started")
        print("auto-check passed")
        for p, sims in ((1, SEARCH_SIMS), (2, 0), (2, SEARCH_SIMS)):
            spec, fields = sft_spec(p, sims)
            print(f"\n== {spec['name']}")
            extend(spec, results, ROOT, opponents, games_per_opponent=GAMES, batch_games=BATCH, fields=fields, gpu=GPU)
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)

## 3. CNN RL from CNN SFT pass 1 (Step 18)
AlphaZero's training with its settings unchanged, starting from pass 1's weights, until CNN SFT pass 1 + RL reaches
19.3 GPU-h. Resumable (rerun to continue). One row per round in `az_log.csv`.

In [ ]:
from pathlib import Path
from train.az_train import list_checkpoints as az_checkpoints, train as train_az

AZ = dataclasses.replace(AZ_CFG, init_from=f"cnn_sft/{RUN}@p1", tag="cnnsft_p1" + ("_smoke" if SMOKE else ""))
AZ_MCTS = MCTS
if SMOKE:
    AZ = dataclasses.replace(AZ, games_per_round=4, max_plies=40, buffer_positions=5_000, checkpoint_every=1)
    AZ_MCTS = dataclasses.replace(MCTS, simulations=8)
AZ_RUN = az_run_name(AZ, NET)
AZ_DIR = os.path.join(ROOT, "checkpoints", "alphazero", AZ_RUN)
init_h = sft_gpu_hours_at(RUN_DIR, 1)
print(f"RL run: {AZ_RUN} | start: CNN SFT pass 1 ({init_h:.3f} GPU-h) | total budget {RL_TOTAL_GPU_H} GPU-h")
summary = train_az(AZ, NET, AZ_MCTS, AZ_DIR, max_gpu_hours=None if SMOKE else RL_TOTAL_GPU_H,
                   max_rounds=2 if SMOKE else None, init_dir=Path(RUN_DIR) / "checkpoints" / "pass_0001",
                   init_gpu_hours=init_h, extra_info={"gpu": GPU})
print(json.dumps(summary, indent=2))

## 4. Rate the RL model's final checkpoint, with and without search

In [ ]:
from train.az_train import gpu_hours_at as az_gpu_hours_at

if SMOKE:
    print("smoke: evals skipped -- smoke test passed")
else:
    final = az_checkpoints(AZ_DIR)[-1][0]
    base = {"step": final, "seed": 0, "train_gpu_hours": round(az_gpu_hours_at(AZ_DIR, final), 4),
            "label_cpu_hours": LABEL_CPU, "dataset_size": json.load(open(os.path.join(RUN_DIR, "run_config.json")))["train_rows"]}
    SPECS = [({"name": f"{AZ_RUN}_nosearch@r{final}", "kind": "alphazero", "run": AZ_RUN, "step": final, "simulations": 0},
              {**base, "run": f"alphazero/{AZ_RUN}_nosearch", "note": "CNN RL from CNN SFT pass 1 (Step 18), no search"}),
             ({"name": f"{AZ_RUN}@r{final}", "kind": "alphazero", "run": AZ_RUN, "step": final, "simulations": SEARCH_SIMS},
              {**base, "run": f"alphazero/{AZ_RUN}", "note": f"CNN RL from CNN SFT pass 1 (Step 18), {SEARCH_SIMS}-sim search"})]
    results = ResultsTable(results_path)
    opponents = build_reference_agents(results, PANEL, ROOT)
    try:
        for spec, fields in SPECS:
            print(f"\n== {spec['name']}")
            extend(spec, results, ROOT, opponents, games_per_opponent=GAMES, batch_games=BATCH, fields=fields, gpu=GPU)
    finally:
        close_agents(opponents)

## 5. The CNN results next to the rest (all on the full panel)

In [ ]:
import pandas as pd

if not SMOKE:
    t = ResultsTable(results_path)
    names = [f"{RUN}_nosearch@p1", f"{RUN}@p1", f"{RUN}_nosearch@p2", f"{RUN}@p2", SPECS[0][0]["name"], SPECS[1][0]["name"],
             "az_b6c64_s0_nosearch@r109", "az_b6c64_s0@r109",
             "structured_masked_nall_e2_s0_scaled_d12_n500000@62500", "structured_masked_nall_e2_s0_scaled_d12_n500000_mcts16@62500"]
    rows = [t.rows[n] for n in names if n in t.rows]
    display(pd.DataFrame(rows)[["name", "role", "rating", "ci_low", "ci_high", "games", "wins", "draws", "losses",
                                "train_gpu_hours", "sec_per_move"]])